
1. **Nhánh Mặt (`FaceCNN`)**: `ResNet-18` cải tiến **Stem `3x3 stride 1` (bỏ maxpool)** giúp ảnh mặt `48x48` giữ nguyên độ phân giải đặc trưng `6x6` (thay vì bị co về `2x2`), tích hợp **Squeeze-and-Excitation (SE Attention)** + `Dropout(0.4)`.
2. **Nhánh Tiếng (`AudioResNetMel`)**: Tự động trích xuất **3 kênh `[Mel, ΔMel, ΔΔMel]` (`128x128`)** đưa qua `ResNet-18 Pretrained` kết hợp **Attentive Statistics Pooling (Mean + Std theo trọng số thời gian)** + `Dropout(0.4)`.
3. **Chống Overfitting**:
   - **Augmentation mạnh**: Ảnh mặt (`HorizontalFlip`, `ShiftScaleRotate ±15°`, `ColorJitter`, `RandomErasing p=0.25`); Âm thanh (`Time-shift ±0.35s`, `Random Gain ±5dB`, `Gaussian Noise SNR 18-30dB`, `SpecAugment 2 masks`).
   - **Mixup ($\alpha = 0.2$)** trộn cặp mẫu ngẫu nhiên làm mượt biên quyết định.
   - **Smoothed Inverse-Frequency Class Weights + Label Smoothing ($\varepsilon = 0.1$)**.
   - **Model EMA (Exponential Moving Average)** làm mượt trọng số đánh giá.
   - **Đóng băng Backbone 2 epoch đầu + Layer-wise LR (`lr_backbone = 0.3 * lr_head`) + Warmup Cosine Annealing**.
   - **GroupKFold theo `video_id`** chống rò rỉ dữ liệu người nói/cảnh quay.
4. **Late Fusion có Hiệu chỉnh Nhiệt độ (`Temperature Scaling`) + Tìm trọng số tối ưu trên tập Val + Test-Time Augmentation (TTA)**.

## Bước 1: Kiểm tra GPU & Cài đặt thư viện

In [ ]:
!nvidia-smi
!pip install -q torch torchvision librosa soundfile scikit-learn albumentations pyyaml pandas matplotlib

: 

## Bước 2: Giải nén dữ liệu Tiếng Việt (`vn_youtube_data.zip`) hoặc Tải tự động RAVDESS từ Zenodo

- Nếu bạn đã tải file `vn_youtube_data.zip` từ notebook cào dữ liệu, hãy upload lên Colab ở ô bên dưới.
- Nếu chưa có, ô bên dưới cũng hỗ trợ tải tự động bộ **RAVDESS Speech (24 diễn viên)** từ Zenodo để pre-train nhánh âm thanh.

In [ ]:
import zipfile
from pathlib import Path

if Path("vn_youtube_data.zip").exists():
    with zipfile.ZipFile("vn_youtube_data.zip", "r") as zf:
        zf.extractall(".")
    print("✅ Đã giải nén vn_youtube_data.zip thành công!")
else:
    print("ℹ️ Chưa thấy vn_youtube_data.zip. Bạn có thể upload vn_youtube_data.zip vào cột Files bên trái của Colab.")

## Bước 3: Định nghĩa Kiến trúc Mô hình (`FaceCNN` Stem 3×3 + `AudioResNetMel` 3 kênh + `ModelEMA` + `Mixup`)

In [ ]:
import copy
import json
import math
import random
import time
import librosa
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

EMOTIONS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(8, channels // reduction)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, mid, bias=False), nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x).unsqueeze(-1).unsqueeze(-1)

class FaceCNN(nn.Module):
    """ResNet-18 grayscale với stem 3x3 stride 1 (bỏ maxpool) + SEBlock + Dropout chống overfit."""
    def __init__(self, num_classes: int = 7, pretrained: bool = True, dropout: float = 0.4):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        old_w = backbone.conv1.weight.data.mean(dim=1, keepdim=True)
        backbone.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        with torch.no_grad():
            backbone.conv1.weight.copy_(F.interpolate(old_w, size=(3, 3), mode="bilinear", align_corners=True))
        backbone.maxpool = nn.Identity()
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.se = SEBlock(in_features, 16)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, num_classes))

    def forward(self, x):
        b = self.backbone
        x = b.layer4(b.layer3(b.layer2(b.layer1(b.maxpool(b.relu(b.bn1(b.conv1(x))))))))
        x = b.avgpool(self.se(x))
        return self.head(torch.flatten(x, 1))

    def set_backbone_trainable(self, trainable: bool = True):
        for name, p in self.backbone.named_parameters():
            if not name.startswith("layer4"):
                p.requires_grad = trainable

    def get_param_groups(self, lr: float, weight_decay: float, backbone_lr_mult: float = 0.3):
        bp = [p for p in self.backbone.parameters() if p.requires_grad]
        hp = [p for p in list(self.se.parameters()) + list(self.head.parameters()) if p.requires_grad]
        return [{"params": bp, "lr": lr * backbone_lr_mult, "weight_decay": weight_decay},
                {"params": hp, "lr": lr, "weight_decay": weight_decay}]

class AttentiveStatsPooling(nn.Module):
    def __init__(self, in_dim: int, bottleneck: int = 128):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv1d(in_dim, bottleneck, 1), nn.Tanh(),
            nn.Conv1d(bottleneck, in_dim, 1), nn.Softmax(dim=-1)
        )
    def forward(self, x):
        w = self.attn(x)
        mean = torch.sum(w * x, dim=-1)
        std = torch.sqrt(torch.sum(w * ((x - mean.unsqueeze(-1)) ** 2), dim=-1).clamp_min(1e-6))
        return torch.cat([mean, std], dim=1)

class AudioResNetMel(nn.Module):
    """ResNet-18 Pretrained trên 3 kênh [Mel, ΔMel, ΔΔMel] + AttentiveStatsPooling + Dropout."""
    def __init__(self, num_classes: int = 7, pretrained: bool = True, dropout: float = 0.4):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity(); backbone.avgpool = nn.Identity()
        self.backbone = backbone
        self.pool = AttentiveStatsPooling(in_features, 128)
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_features * 2), nn.Dropout(dropout),
            nn.Linear(in_features * 2, 128), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(128, num_classes)
        )

    def forward(self, x):
        if x.shape[1] == 1:
            p1 = F.pad(x, (1, 1, 0, 0), mode="replicate")
            d1 = (p1[:, :, :, 2:] - p1[:, :, :, :-2]) * 0.5
            p2 = F.pad(d1, (1, 1, 0, 0), mode="replicate")
            d2 = (p2[:, :, :, 2:] - p2[:, :, :, :-2]) * 0.5
            x = torch.cat([x, d1, d2], dim=1)
        b = self.backbone
        x = b.layer4(b.layer3(b.layer2(b.layer1(b.maxpool(b.relu(b.bn1(b.conv1(x))))))))
        return self.head(self.pool(x.mean(dim=2)))

    def set_backbone_trainable(self, trainable: bool = True):
        for name, p in self.backbone.named_parameters():
            if not name.startswith("layer4"):
                p.requires_grad = trainable

    def get_param_groups(self, lr: float, weight_decay: float, backbone_lr_mult: float = 0.3):
        bp = [p for p in self.backbone.parameters() if p.requires_grad]
        hp = [p for p in list(self.pool.parameters()) + list(self.head.parameters()) if p.requires_grad]
        return [{"params": bp, "lr": lr * backbone_lr_mult, "weight_decay": weight_decay},
                {"params": hp, "lr": lr, "weight_decay": weight_decay}]


## Bước 4: Chạy Huấn luyện 2 Nhánh + Đánh giá Late Fusion (Kèm Temperature Scaling & TTA)

In [ ]:
# Nếu chạy trong thư mục dự án DAKHDL đã clone hoặc đã giải nén vn_youtube_data.zip:
import subprocess, sys
if Path("src/training/__main__.py").exists() and Path("data/splits/vn_face_train.csv").exists():
    subprocess.run([sys.executable, "-m", "src.training", "--branch", "face", "--splits-prefix", "vn_", "--epochs", "25"], check=True)
    subprocess.run([sys.executable, "-m", "src.training", "--branch", "audio", "--splits-prefix", "vn_", "--epochs", "25"], check=True)
    subprocess.run([
        sys.executable, "-m", "src.evaluation",
        "--face-ckpt", "models/checkpoints/face/best.pt",
        "--audio-ckpt", "models/checkpoints/audio/best.pt",
        "--splits-prefix", "vn_", "--tta"
    ], check=True)
else:
    print("Hãy đảm bảo đã có thư mục data/splits/ (từ vn_youtube_data.zip hoặc make_splits.py) để bắt đầu huấn luyện!")